# DogFacenet V2

Esse notebook foi feito para o treinamento da rede siamesa, que corresponde à versão `.py` deste mesmo arquivo

#### Imports

In [1]:
from src.dataset import TripletGenerator
from src.model import get_embedding_module, get_siamese_network, SiameseModel
from src.dataset import MapFunction
from tensorflow import keras
import tensorflow as tf
import os

2025-10-22 10:18:36.361868: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-22 10:18:36.362108: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-22 10:18:36.409385: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-22 10:18:37.598999: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

#### Configs

In [2]:
# path to training and testing data
TRAIN_DATASET = "../data/train"
TEST_DATASET = "../data/test"

# model input image size
IMAGE_SIZE = (224, 224)

# batch size and the buffer size
BATCH_SIZE = 256
BUFFER_SIZE = BATCH_SIZE * 2    # determines the number of elements from which the next 
                                # element for the shuffled dataset is randomly drawn

# define autotune
AUTO = tf.data.AUTOTUNE

# define the training parameters
LEARNING_RATE = 0.0001
STEPS_PER_EPOCH = 1
VALIDATION_STEPS = 1
EPOCHS = 1

# define the path to save the model
OUTPUT_PATH = "models"
MODEL_PATH = os.path.join(OUTPUT_PATH, "siamese_network.keras")

#### Training

In [3]:
# create the data input pipeline for train and val dataset
print("[INFO] building the train and validation generators...")
trainTripletGenerator = TripletGenerator(
	datasetPath=TRAIN_DATASET)
valTripletGenerator = TripletGenerator(
	datasetPath=TRAIN_DATASET)

print("[INFO] building the train and validation `tf.data` dataset...")
trainTfDataset = tf.data.Dataset.from_generator(
	generator=trainTripletGenerator.get_next_element,
	output_signature=(
		tf.TensorSpec(shape=(), dtype=tf.string),
		tf.TensorSpec(shape=(), dtype=tf.string),
		tf.TensorSpec(shape=(), dtype=tf.string),
	)
)
valTfDataset = tf.data.Dataset.from_generator(
	generator=valTripletGenerator.get_next_element,
	output_signature=(
		tf.TensorSpec(shape=(), dtype=tf.string),
		tf.TensorSpec(shape=(), dtype=tf.string),
		tf.TensorSpec(shape=(), dtype=tf.string),
	)
)

[INFO] building the train and validation generators...
[INFO] building the train and validation `tf.data` dataset...


2025-10-22 10:21:16.547423: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [4]:
# preprocess the images
mapFunction = MapFunction(imageSize=IMAGE_SIZE)
print("[INFO] building the train and validation `tf.data` pipeline...")
trainDs = (trainTfDataset
    .map(mapFunction)
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE)
    .prefetch(AUTO)
)
valDs = (valTfDataset
    .map(mapFunction)
    .batch(BATCH_SIZE)
    .prefetch(AUTO)
)

[INFO] building the train and validation `tf.data` pipeline...


In [5]:
# build the embedding module and the siamese network
print("[INFO] build the siamese model...")
embeddingModule = get_embedding_module(imageSize=IMAGE_SIZE)
siameseNetwork =  get_siamese_network(
	imageSize=IMAGE_SIZE,
	embeddingModel=embeddingModule,
)
siameseModel = SiameseModel(
	siameseNetwork=siameseNetwork,
	margin=0.5,
	lossTracker=keras.metrics.Mean(name="loss"),
)

# compile the siamese model
siameseModel.compile(
	optimizer=keras.optimizers.Adam(LEARNING_RATE)
)

[INFO] build the siamese model...


In [6]:
# train and validate the siamese model
print("[INFO] training the siamese model...")
siameseModel.fit(
	trainDs,
	steps_per_epoch=STEPS_PER_EPOCH,
	validation_data=valDs,
	validation_steps=VALIDATION_STEPS,
	epochs=EPOCHS,
)

# create output directory
if not os.path.exists(OUTPUT_PATH):
	os.makedirs(OUTPUT_PATH)

# save the siamese network to disk
modelPath = MODEL_PATH
print(f"[INFO] saving the siamese network to {modelPath}...")
keras.models.save_model(
	model=siameseModel.siameseNetwork,
	filepath=modelPath,
	include_optimizer=False,
)

[INFO] training the siamese model...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32s/step - loss: 0.4717

2025-10-22 10:23:46.837585: W tensorflow/core/kernels/data/prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 462422016 bytes after encountering the first element of size 462422016 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


1/1 ━━━━━━━━━━━━━━━━━━━━ 64s 64s/step - loss: 0.4717 - val_loss: 0.4701
[INFO] saving the siamese network to models/siamese_network.keras...
